## Notebook 06 -- Chapter 5 figures

Rebuilds Fig 5.12 (RMSE by model), Fig 5.13 (P007 session 9 traces) and Fig 5.14 (MLP per-fold RMSE).
PNGs (300 dpi) go to `Writeups/Images/`. Style: SciencePlots `science` + `no-latex`.

**Aggregation rule (session mean).** Per-fold RMSE = `metrics['overall']['rmse']`. MLP and XGBoost
LOSO values are the mean over seeds 0-19 per fold first. Participant value = mean of that participant's
fold RMSEs. Cohort = mean of the 10 participant values. SD = sample SD (ddof=1) across the 10 participants.

In [ ]:
from _common import *
import matplotlib.pyplot as plt
import scienceplots
plt.style.use(["science", "no-latex"])

OUT_DIR = Path(r"C:\Users\Guillaume\Documents\DLSU\THESIS\Writeups\Images")
assert OUT_DIR.exists()
PARTICIPANTS = list(range(1, 11))
colors = plt.rcParams['axes.prop_cycle'].by_key()['color']

### Load fold metrics and apply the aggregation rule

Sources, all in `notebooks/`: `fold_metrics_ridge.json` (LOSO, single run), `fold_metrics_ridge_rung3.json`
(LOPO, single run), `fold_metrics_xgboost_seeds.json` (arm `rung1`, seeds 0-19), `fold_metrics_xgboost_rung3.json`
(LOPO, single seed), `fold_metrics_mlp_seeds.json` (arms `rung1` and `rung3`, seeds 0-19).

In [ ]:
def load_json(name):
    with open(CURRENT_FOLDER / name) as f:
        return json.load(f)

ridge_loso = load_json('fold_metrics_ridge.json')
ridge_lopo = load_json('fold_metrics_ridge_rung3.json')
xgb_seeds = load_json('fold_metrics_xgboost_seeds.json')
xgb_lopo = load_json('fold_metrics_xgboost_rung3.json')
mlp_seeds = load_json('fold_metrics_mlp_seeds.json')

assert len(ridge_loso) == 98 and len(ridge_lopo) == 10 and len(xgb_lopo) == 10
assert len(xgb_seeds) == 98 * 20
assert sorted({r['seed'] for r in xgb_seeds}) == list(range(20))
assert sorted({r['seed'] for r in mlp_seeds}) == list(range(20))

In [ ]:
# --- per-fold RMSE, one value per (participant, session) fold ---
# Ridge LOSO: single run
fold_ridge = {}
for r in ridge_loso:
    fold_ridge[(r['participant_id'], r['test_session'])] = r['metrics']['overall']['rmse']

# XGBoost LOSO: mean over 20 seeds per fold
tmp = {}
for r in xgb_seeds:
    assert r['arm'] == 'rung1'
    tmp.setdefault((r['participant_id'], r['test_session']), []).append(r['metrics']['overall']['rmse'])
fold_xgb = {}
for key, vals in tmp.items():
    assert len(vals) == 20
    fold_xgb[key] = float(np.mean(vals))

# MLP LOSO: mean over 20 seeds per fold
tmp = {}
for r in mlp_seeds:
    if r['arm'] == 'rung1':
        tmp.setdefault((r['participant_id'], r['test_session']), []).append(r['metrics']['overall']['rmse'])
fold_mlp = {}
for key, vals in tmp.items():
    assert len(vals) == 20
    fold_mlp[key] = float(np.mean(vals))

assert len(fold_ridge) == len(fold_xgb) == len(fold_mlp) == 98
assert set(fold_ridge) == set(fold_xgb) == set(fold_mlp)

# --- LOSO participant values: mean of that participant's fold RMSEs ---
loso = {'Ridge': {}, 'XGBoost': {}, 'MLP': {}}
for p in PARTICIPANTS:
    for name, fold_dict in [('Ridge', fold_ridge), ('XGBoost', fold_xgb), ('MLP', fold_mlp)]:
        vals = [v for (pp, s), v in fold_dict.items() if pp == p]
        loso[name][p] = float(np.mean(vals))

# --- LOPO participant values: one fold per participant ---
# The LOPO files hold one pooled RMSE per held-out participant (all frames). Session mean needs a value per
# held-out session. Ridge and XGBoost: fold_metrics_rung4.json stores the frozen LOPO model's per-session
# RMSE ('rmse_rung3') for every session, repeated across enrolment turns (deduplicated here).
# MLP: the 'test' list of each rung3 record in fold_metrics_mlp_seeds.json, mean over 20 seeds per session.
rung4 = load_json('fold_metrics_rung4.json')
sess_lopo = {'Ridge': {}, 'XGBoost': {}, 'MLP': {}}      # model -> (participant, session) -> rmse
sess_n = {}
for r in rung4:
    if r['model'] == 'MLP':
        continue
    for t in r['test']:
        key = (r['participant_id'], t['session'])
        if key in sess_lopo[r['model']]:
            assert np.isclose(sess_lopo[r['model']][key], t['rmse_rung3'])
        sess_lopo[r['model']][key] = t['rmse_rung3']
        sess_n[(r['model'], key)] = t['n']
tmp = {}
for r in mlp_seeds:
    if r['arm'] == 'rung3':
        for t in r['test']:
            tmp.setdefault((r['participant_id'], t['session']), []).append(t['rmse'])
for key, vals in tmp.items():
    assert len(vals) == 20
    sess_lopo['MLP'][key] = float(np.mean(vals))

# guard: frame-pooling the per-session values must rebuild the stored pooled LOPO RMSE (Ridge, XGBoost)
for name, records in [('Ridge', ridge_lopo), ('XGBoost', xgb_lopo)]:
    for r in records:
        p = r['participant_id']
        keys = [k for k in sess_lopo[name] if k[0] == p]
        pooled = np.sqrt(sum(sess_n[(name, k)] * sess_lopo[name][k] ** 2 for k in keys) / sum(sess_n[(name, k)] for k in keys))
        assert abs(pooled - r['metrics']['overall']['rmse']) < 1e-6, (name, p)

lopo = {'Ridge': {}, 'XGBoost': {}, 'MLP': {}}
for name in lopo:
    for p in PARTICIPANTS:
        vals = [v for (pp, s), v in sess_lopo[name].items() if pp == p]
        assert len(vals) == len([k for k in fold_ridge if k[0] == p])      # every session of the participant
        lopo[name][p] = float(np.mean(vals))
for name in lopo:
    assert sorted(lopo[name]) == PARTICIPANTS and sorted(loso[name]) == PARTICIPANTS

In [ ]:
# --- cohort mean and SD (ddof=1) across the 10 participant values ---
summary = {}
for name in ['Ridge', 'XGBoost', 'MLP']:
    a = np.array([loso[name][p] for p in PARTICIPANTS])
    b = np.array([lopo[name][p] for p in PARTICIPANTS])
    summary[name] = {'loso_mean': a.mean(), 'loso_sd': a.std(ddof=1), 'lopo_mean': b.mean(), 'lopo_sd': b.std(ddof=1)}
    print(f"{name:8s} LOSO {a.mean():.2f} ({a.std(ddof=1):.2f})   LOPO {b.mean():.2f} ({b.std(ddof=1):.2f})")

### Verification against Tables 5.9 / 5.10 and the per-participant MLP LOSO values

In [ ]:
expected = {
    'Ridge':   {'loso': (7.14, 1.15), 'lopo': (10.16, 1.49)},
    'XGBoost': {'loso': (6.30, 1.07), 'lopo': (9.76, 1.45)},
    'MLP':     {'loso': (6.45, 1.32), 'lopo': (9.28, 1.60)},
}
for name in expected:
    for arm in ['loso', 'lopo']:
        got_mean = round(summary[name][arm + '_mean'], 2)
        got_sd = round(summary[name][arm + '_sd'], 2)
        print(name, arm, 'got', got_mean, got_sd, 'expected', expected[name][arm])
        assert (got_mean, got_sd) == expected[name][arm]

expected_p = {1: 6.90, 5: 3.94, 6: 9.08, 9: 6.29}
for p, v in expected_p.items():
    print('MLP LOSO P%03d' % p, round(loso['MLP'][p], 2), 'expected', v)
    assert round(loso['MLP'][p], 2) == v
print('all Table 5.9 / 5.10 checks passed')

### Fig 5.12 -- RMSE by model and evaluation

Grouped bars per model (Ridge, XGBoost, MLP): cohort mean of the 10 participant values for Cross-mount (LOSO)
and Cross-participant (LOPO). Error bar = 1 SD (ddof=1) across participants. Dots = the 10 participant values.
Data: `fold_metrics_ridge*.json`, `fold_metrics_xgboost_seeds.json`, `fold_metrics_xgboost_rung3.json`,
`fold_metrics_mlp_seeds.json` (20-seed means for MLP and XGBoost LOSO, MLP LOPO).

In [ ]:
models = ['Ridge', 'XGBoost', 'MLP']
rng = np.random.default_rng(0)
width = 0.36
x = np.arange(len(models))

fig, ax = plt.subplots(figsize=(7, 5))
for i, name in enumerate(models):
    for j, (arm, label, vals) in enumerate([('loso', 'Cross-mount (LOSO)', loso[name]), ('lopo', 'Cross-participant (LOPO)', lopo[name])]):
        xc = x[i] + (j - 0.5) * width
        mean = summary[name][arm + '_mean']
        sd = summary[name][arm + '_sd']
        ax.bar(xc, mean, width * 0.92, color=colors[j], edgecolor='black', linewidth=0.5,
               alpha=0.75, label=label if i == 0 else None)
        ax.errorbar(xc, mean, yerr=sd, color='black', capsize=4, lw=1)
        pv = np.array([vals[p] for p in PARTICIPANTS])
        jitter = rng.uniform(-width * 0.25, width * 0.25, size=len(pv))
        ax.scatter(xc + jitter, pv, s=10, color=colors[j], edgecolor='black', linewidth=0.5, zorder=3)
ax.set_xticks(x)
ax.set_xticklabels(models)
ax.set_ylabel('RMSE (%MVC)')
ax.set_ylim(0, 15)
ax.legend(loc='upper left')
ax.grid(axis='y', linestyle='--', color='gray', alpha=0.5)
fig.savefig(OUT_DIR / 'fig5_12_rmse_by_model.png', dpi=300, bbox_inches='tight')
plt.show()

### Fig 5.14 -- MLP Cross-mount (LOSO) per-fold RMSE

All 98 folds, MLP, mean over seeds 0-19 per fold. Strip plot by participant (jittered), black bar = participant
mean, grey dashed line = cohort mean (mean of the 10 participant values). Data: `fold_metrics_mlp_seeds.json`, arm `rung1`.

In [ ]:
# verify Table 5.14 numbers first
p006 = [v for (p, s), v in fold_mlp.items() if p == 6]
p009 = [v for (p, s), v in fold_mlp.items() if p == 9]
p006_max = max(p006)
p009_range = max(p009) - min(p009)
print('P006 max', round(p006_max, 2), 'expected 27.46')
print('P009 range', round(p009_range, 2), 'expected 2.25')
assert round(p006_max, 2) == 27.46
assert round(p009_range, 2) == 2.25
p006_s4 = fold_mlp[(6, 4)]
assert round(p006_s4, 2) == 27.46

In [ ]:
rng = np.random.default_rng(1)
cohort_mean = summary['MLP']['loso_mean']

fig, ax = plt.subplots(figsize=(7, 4))
for k, p in enumerate(PARTICIPANTS):
    vals = np.array([v for (pp, s), v in sorted(fold_mlp.items()) if pp == p])
    jitter = rng.uniform(-0.18, 0.18, size=len(vals))
    ax.scatter(k + jitter, vals, s=14, color=colors[0], alpha=0.8, edgecolor='none', zorder=2)
    ax.hlines(loso['MLP'][p], k - 0.3, k + 0.3, color='black', lw=2, zorder=3)
ax.axhline(cohort_mean, color='grey', ls='--', lw=1, label='Cohort mean (%.2f)' % cohort_mean)
ax.annotate('P006 S4', xy=(5.1, p006_s4), xytext=(6.2, p006_s4 - 1.0), arrowprops=dict(arrowstyle='-', lw=0.6), fontsize=8)
ax.set_xticks(range(10))
ax.set_xticklabels(['P%03d' % p for p in PARTICIPANTS], rotation=45)
ax.set_ylabel('RMSE (%MVC)')
ax.set_ylim(0, None)
ax.legend(loc='upper left')
ax.grid(axis='y', linestyle='--', color='gray', alpha=0.3)
fig.savefig(OUT_DIR / 'fig5_14_mlp_fold_rmse.png', dpi=300, bbox_inches='tight')
plt.show()

### Fig 5.13 -- P007 session 9, predicted vs ground-truth force

Per-frame predictions are not stored, so two MLPs are retrained (seed 42, the stored best hyperparameters):
Cross-mount (LOSO) = fold with test_session 9 of participant 7, trained on P007's other sessions
(`fold_metrics_mlp.json`); Cross-participant (LOPO) = trained on the other 9 participants
(`fold_metrics_mlp_rung3.json`). Training code is copied from `05_mlp.ipynb`: `StandardScaler` + MLP,
`FEATURE_COLS`, target `pct_mvc_corr` (%MVC), fixed `n_epochs`, `torch.manual_seed(42)`, no early stopping.
Prediction in kg = %MVC / 100 x P007 MVC, where MVC = `force_kg_corr` / `pct_mvc_corr` x 100 (constant within P007).
Ground truth = `force_kg_corr`. Time = (`timestamp_ms` minus session start) / 1000.
Rows are those of `FEATURES`, i.e. the first frame of each trial is dropped (velocity NaN).

In [ ]:
import torch
import torch.nn as nn
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('DEVICE:', DEVICE)

class MLPTorch(nn.Module):
    def __init__(self, n_features, hidden_dims, dropout):
        super().__init__()
        layers = []
        prev = n_features
        for h in hidden_dims:
            layers += [nn.Linear(prev, h), nn.ReLU(), nn.Dropout(dropout)]
            prev = h
        layers += [nn.Linear(prev, 1)]
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x).squeeze(-1)


class MLPRegressor:
    def __init__(self, hidden_dims, dropout, lr, weight_decay, batch_size, n_epochs, random_state=RANDOM_SEED):
        self.hidden_dims = hidden_dims
        self.dropout = dropout
        self.lr = lr
        self.weight_decay = weight_decay
        self.batch_size = batch_size
        self.n_epochs = n_epochs
        self.random_state = random_state

    def fit(self, X, y):
        torch.manual_seed(self.random_state)
        X = np.asarray(X, dtype=np.float32)
        y = np.asarray(y, dtype=np.float32)
        self.model_ = MLPTorch(X.shape[1], self.hidden_dims, self.dropout).to(DEVICE)
        opt = torch.optim.Adam(self.model_.parameters(), lr=self.lr, weight_decay=self.weight_decay)
        loss_fn = nn.MSELoss()
        X_t = torch.from_numpy(X).to(DEVICE)
        y_t = torch.from_numpy(y).to(DEVICE)
        n = len(X_t)
        self.model_.train()
        for epoch in range(self.n_epochs):
            perm = torch.randperm(n, device=DEVICE)
            for start in range(0, n, self.batch_size):
                idx = perm[start:start + self.batch_size]
                opt.zero_grad()
                pred = self.model_(X_t[idx])
                loss = loss_fn(pred, y_t[idx])
                loss.backward()
                opt.step()
        return self

    def predict(self, X):
        X = np.asarray(X, dtype=np.float32)
        X_t = torch.from_numpy(X).to(DEVICE)
        self.model_.eval()
        with torch.no_grad():
            pred = self.model_(X_t)
        return pred.cpu().numpy()


def hidden_dims_from(width, n_layers):
    dims = []
    w = width
    for _ in range(n_layers):
        dims.append(max(w, 8))
        w = max(w // 2, 8)
    return dims


def make_model(params):
    hidden_dims = hidden_dims_from(params['width'], params['n_layers'])
    return Pipeline([
        ('scale', StandardScaler()),
        ('mlp', MLPRegressor(
            hidden_dims=hidden_dims,
            dropout=params['dropout'],
            lr=params['lr'],
            weight_decay=params['weight_decay'],
            batch_size=params['batch_size'],
            n_epochs=params['n_epochs'],
            random_state=RANDOM_SEED,
        )),
    ])

In [ ]:
P, S = 7, 9
stored_loso = {(r['participant_id'], r['test_session']): r for r in load_json('fold_metrics_mlp.json')}
stored_lopo = {r['participant_id']: r for r in load_json('fold_metrics_mlp_rung3.json')}

# --- LOSO: P007, test session 9, train on P007's other sessions ---
fold = next(f for f in SESSION_FOLDS if f['participant_id'] == P and f['test_session'] == S)
params_loso = stored_loso[(P, S)]['best_params']
train_mask = (FEATURES['participant_id'] == P) & (FEATURES['session_id'].isin(fold['train_sessions']))
test_mask = (FEATURES['participant_id'] == P) & (FEATURES['session_id'] == S)
assert not (FEATURES.loc[train_mask, 'session_id'] == S).any()
assert set(FEATURES.loc[train_mask, 'session_id'].unique()) == set(fold['train_sessions'])

model = make_model(params_loso)
model.fit(FEATURES.loc[train_mask, FEATURE_COLS], FEATURES.loc[train_mask, 'pct_mvc_corr'])
pred_loso_pct = np.asarray(model.predict(FEATURES.loc[test_mask, FEATURE_COLS]), dtype=float)
y_pct = FEATURES.loc[test_mask, 'pct_mvc_corr'].to_numpy()
rmse_loso_retrain = float(np.sqrt(np.mean((pred_loso_pct - y_pct) ** 2)))
del model
torch.cuda.empty_cache()

In [ ]:
# --- LOPO: train on the other 9 participants, test on P007 ---
sfold = next(f for f in SUBJECT_FOLDS if f['test_participant'] == 'participant%03d' % P)
train_ps = [int(p.removeprefix('participant')) for p in sfold['train_participants']]
assert P not in train_ps and len(train_ps) == 9
params_lopo = stored_lopo[P]['best_params']
pool_mask = FEATURES['participant_id'].isin(train_ps)
held_mask = FEATURES['participant_id'] == P
assert not (FEATURES.loc[pool_mask, 'participant_id'] == P).any()

model = make_model(params_lopo)
model.fit(FEATURES.loc[pool_mask, FEATURE_COLS], FEATURES.loc[pool_mask, 'pct_mvc_corr'])
pred_lopo_all = np.asarray(model.predict(FEATURES.loc[held_mask, FEATURE_COLS]), dtype=float)
s9_in_held = (FEATURES.loc[held_mask, 'session_id'] == S).to_numpy()
pred_lopo_pct = pred_lopo_all[s9_in_held]
rmse_lopo_s9 = float(np.sqrt(np.mean((pred_lopo_pct - y_pct) ** 2)))
rmse_lopo_p007_all = float(np.sqrt(np.mean((pred_lopo_all - FEATURES.loc[held_mask, 'pct_mvc_corr'].to_numpy()) ** 2)))
del model
torch.cuda.empty_cache()

In [ ]:
# sanity check against stored seed-42 values (fold_metrics_mlp*.json are the seed-42 runs)
stored_loso_s9 = stored_loso[(P, S)]['metrics']['overall']['rmse']
stored_lopo_p7 = stored_lopo[P]['metrics']['overall']['rmse']
print('LOSO P007 S9: retrained %.6f  stored seed 42 %.6f  gap %.6f' % (rmse_loso_retrain, stored_loso_s9, rmse_loso_retrain - stored_loso_s9))
print('LOPO P007 (all sessions): retrained %.6f  stored seed 42 %.6f  gap %.6f' % (rmse_lopo_p007_all, stored_lopo_p7, rmse_lopo_p007_all - stored_lopo_p7))
print('LOPO P007 S9 only (plotted): %.4f' % rmse_lopo_s9)
print('LOSO P007 S9 20-seed mean %.4f' % fold_mlp[(P, S)])
assert abs(rmse_loso_retrain - stored_loso_s9) < 0.05
assert abs(rmse_lopo_p007_all - stored_lopo_p7) < 0.05

In [ ]:
# convert %MVC to kg, build time axis, plot
d = FEATURES.loc[test_mask]
mvc_all = (d['force_kg_corr'] / d['pct_mvc_corr'] * 100).to_numpy()
mvc_all = mvc_all[d['pct_mvc_corr'].to_numpy() > 1]
assert mvc_all.max() - mvc_all.min() < 1e-6
mvc_kg = float(np.median(mvc_all))
print('P007 MVC (kg):', round(mvc_kg, 3))

t = d['timestamp_ms'].to_numpy()
t = (t - t.min()) / 1000.0   # timestamp_ms is in milliseconds
print('frames:', len(t), ' duration (s):', round(t.max(), 1))
truth_kg = d['force_kg_corr'].to_numpy()
loso_kg = pred_loso_pct / 100 * mvc_kg
lopo_kg = pred_lopo_pct / 100 * mvc_kg

# trim the triple-tap bookends: opening taps end ~1.5 s, closing taps start ~132.7 s
# (last grip released ~131.0 s). Window re-zeroed so the plot starts at 0 s.
T_START, T_END = 2.5, 132.5
keep = (t >= T_START) & (t <= T_END)
t_plot = t[keep] - T_START
print('plotted window (s):', T_START, '-', T_END, ' frames:', keep.sum())

fig, ax = plt.subplots(figsize=(10, 3.4))
ax.plot(t_plot, truth_kg[keep], lw=0.8, label='Ground truth')
ax.plot(t_plot, loso_kg[keep], ls='--', lw=0.7, label='Cross-mount (LOSO)')
ax.plot(t_plot, lopo_kg[keep], ls='--', lw=0.7, label='Cross-participant (LOPO)')
ax.set_xlabel('Time (s)')
ax.set_ylabel('Force (kg)')
ax.set_xlim(0, T_END - T_START)
ax.set_ylim(-1.5, 31)
ax.legend(loc='upper right')
fig.savefig(OUT_DIR / 'fig5_13_p007_s9_traces.png', dpi=300, bbox_inches='tight')
plt.show()